# simplified_CMB_lite: Colab Starter

This notebook downloads the simplified CMB teaching code from GitHub, compiles the C++ solver, plots a fiducial TT spectrum, and evaluates a Gaussian negative log likelihood for the included Planck-like mock data.

The model parameter vector is

```text
theta = [A_s, omega_cdm, omega_b, H0, n_s, Delta_N_eff]
```

`H0` is in km/s/Mpc. The likelihood also accepts `ell_min` and `ell_max`; the default range is `ell_min=2`, `ell_max=2500`.


In [ ]:
# Download or update the repository.
# If you have edited files inside Colab, rerun from the compile cell below
# rather than rerunning this checkout cell.
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/tsmith2/GGI_cosmology_notebooks.git"
REPO_DIR = Path("/content/GGI_cosmology_notebooks")
PACKAGE_DIR = REPO_DIR / "simplified_CMB_lite"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(PACKAGE_DIR)
commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("Working in", Path.cwd())
print("Git commit", commit)


In [ ]:
# Compile the C++ solver for this Colab runtime.
# This removes any old binary first, which prevents stale macOS/Linux or old-parameter binaries.
subprocess.run(["make", "-C", "cpp", "clean"], check=True)
subprocess.run(["make", "-C", "cpp"], check=True)


In [ ]:
# Import the Python wrapper fresh.
import sys
import inspect
import numpy as np
import matplotlib.pyplot as plt

PYTHON_DIR = str(PACKAGE_DIR / "python")
if PYTHON_DIR not in sys.path:
    sys.path.insert(0, PYTHON_DIR)

print("Python wrapper directory:", PYTHON_DIR)
print("Files there:", sorted(p.name for p in (PACKAGE_DIR / "python").glob("*.py")))

# Colab keeps imported modules in memory.  Removing this entry makes sure the
# notebook uses the file that was just downloaded, not an older in-memory copy.
sys.modules.pop("simplified_cmb_lite_colab", None)

import simplified_cmb_lite_colab as cmb

from simplified_cmb_lite_colab import (
    FID_THETA,
    PARAM_NAMES,
    load_mock_data,
    negative_log_likelihood,
    plot_mock_data_with_residuals,
    plot_power_spectrum,
    tt_spectrum,
)

print("wrapper file:", inspect.getfile(cmb))
print("FID_THETA:", FID_THETA)


In [ ]:
# Choose fiducial values explicitly.
# This is the same as FID_THETA:
# [A_s, omega_cdm, omega_b, H0, n_s, Delta_N_eff]
theta = np.array([2.1e-9, 0.1201, 0.0223, 67.0, 0.965, 0.0])

ell_min = 2
ell_max = 2500
data = load_mock_data()

print("parameter names:", PARAM_NAMES)
print("theta:", theta)
print("low-ell points:", len(data["low_ell"]))
print("Planck-like bins:", len(data["bin_center"]))

if theta[3] < 10.0:
    raise ValueError("theta[3] should be H0 in km/s/Mpc, e.g. 67.0, not h = 0.67")


In [ ]:
# Plot the model spectrum and mock data.
# The residual panel is normalized by the binned cosmic-variance uncertainty.
spec = tt_spectrum(theta, ell_min=ell_min, ell_max=ell_max)
print(f"model peak: ell={int(spec.ell[np.argmax(spec.dell_over_2pi)])}, "
      f"D_ell={np.max(spec.dell_over_2pi):.1f} microK^2")
print(f"mock binned peak: ell={data['bin_center'][np.argmax(data['fake_d_ell_binned'])]:.1f}, "
      f"D_ell={np.max(data['fake_d_ell_binned']):.1f} microK^2")
fig, axes = plot_mock_data_with_residuals(theta, data=data, ell_min=ell_min, ell_max=ell_max)
plt.show()


In [ ]:
# Compute the negative log likelihood for this model.
nll = negative_log_likelihood(theta, data=data, ell_min=ell_min, ell_max=ell_max)
print(f"-log likelihood = {nll:.3f}")


## Try a different model

The next cell changes one parameter, recomputes the TT spectrum, and compares likelihoods.


In [ ]:
theta_alt = theta.copy()
theta_alt[3] = 72.0   # H0 in km/s/Mpc

nll_fid = negative_log_likelihood(theta, data=data, ell_min=ell_min, ell_max=ell_max)
nll_alt = negative_log_likelihood(theta_alt, data=data, ell_min=ell_min, ell_max=ell_max)

print(f"fiducial H0 = {theta[3]:.1f}, -log L = {nll_fid:.3f}")
print(f"alternate H0 = {theta_alt[3]:.1f}, -log L = {nll_alt:.3f}")

fig, ax = plot_power_spectrum(theta, data=data, ell_min=ell_min, ell_max=ell_max)
spec_alt = tt_spectrum(theta_alt, ell_min=ell_min, ell_max=ell_max)
ax.plot(spec_alt.ell, spec_alt.dell_over_2pi, color="royalblue", lw=1.2, label="H0 = 72")
ax.legend(frameon=False)
plt.show()


Notes:

- The first spectrum evaluation may take longer because the code creates a reusable Bessel-function cache.
- Later evaluations reuse that cache.
- This notebook does not run an MCMC sampler; it provides the spectrum and likelihood functions that an MCMC can call.
